# 1118 54일차

## 1. 도구 정의

모델이 직접 계산하지 않고 불러 쓰도록 만들어 둔 함수 (`f31_tool_test`)

모델은 토큰을 이어 붙여 답을 만들어서 자리수가 큰 계산은 틀릴 수 있음. 계산이나 조회는 파이썬 함수에 맡기고, 모델은 어떤 함수에 무엇을 넣을지만 정함

1. `from langchain_core.tools import tool` : 함수를 도구로 바꿔 주는 데코레이터
2. `@tool` : 함수 정의 위에 붙이면 LangChain이 쓰는 도구 객체가 됨
3. 타입 힌트 `aaa: int` : 모델이 넣을 값의 자료형이 됨
4. 독스트링 `""" 두 정수를 더합니다. """` : 이 도구가 무엇을 하는지 모델에게 알려 주는 설명
5. `add.name`, `add.description`, `add.args` : 모델에게 넘어가는 내용을 확인함

- **모델은 함수 본문을 보지 못하고 이름·설명·매개변수만 보고 고름**
- 독스트링이 설명 자리라서, 비워 두면 모델이 어디에 쓸 도구인지 판단할 근거가 없음
- `add.args`는 타입 힌트를 `{'aaa': {'title': 'Aaa', 'type': 'integer'}}` 꼴로 바꿔 놓음
- 매개변수 이름도 모델에게 그대로 넘어가므로, 뜻이 드러나는 이름이 유리함
- `@tool`을 붙인 뒤에는 `add(1, 2)`로 부르지 않고 `add.invoke()`로 부름

## 2. 도구 등록

쓸 수 있는 도구 목록을 모델에게 같이 보내는 것 (`f31_tool_test`, `f32_tool_calls`)

도구를 정의만 해 두면 모델은 그 존재를 모름. 질문을 보낼 때 목록을 함께 보내야 모델이 그중에서 고름

서버도 도구 호출을 받을 수 있게 띄움

```
furiosa-llm serve furiosa-ai/Qwen3-4B-FP8 --devices npu:6 --host 0.0.0.0 --port 8000 --enable-overlap-scheduling --enable-auto-tool-choice --tool-call-parser hermes
```

1. `--enable-auto-tool-choice` : 모델이 도구를 쓸지 스스로 정하게 하는 옵션
2. `--tool-call-parser hermes` : 모델이 적어 내는 도구 호출 형식을 서버가 읽는 방식. Qwen3는 hermes
3. `llm_with_tools = llm.bind_tools([add])` : 도구 목록을 붙인 모델을 새로 받음
4. `llm_with_tools.invoke("237241과 31321을 add로 더해줘")` : 질문을 보냄
5. `response.tool_calls` : 모델이 고른 도구와 넣을 값

- **`bind_tools`는 원래 `llm`을 바꾸지 않고 도구가 붙은 새 객체를 돌려줌**
- 도구를 쓴 응답은 `response.content`가 비고 `tool_calls`에만 내용이 들어옴
- 항목 하나는 `name`(도구 이름)·`args`(넣을 값)·`id`(호출 번호)·`type`이 든 딕셔너리
- `args`의 키가 함수 매개변수 이름(`aaa`, `bbb`)과 같음
- `tool_calls`는 목록이라 도구를 여러 개 고를 수도 있어서 for문으로 꺼냄
- 파서 옵션 없이 띄우면 모델이 형식대로 적어도 서버가 나누지 못해 `content`에 글자로 남음

## 3. 도구 실행

모델이 고른 도구를 실제로 돌려 결과를 얻는 것 (`f33_tool_invoke`)

모델은 어떤 도구에 무엇을 넣을지까지만 정함. 그 값으로 함수를 부르는 건 코드가 함

1. `for call in response.tool_calls:` : 호출을 하나씩 꺼냄
2. `add.invoke(call['args'])` : 꺼낸 `args` 딕셔너리를 그대로 넘겨 도구를 실행함

- **모델이 내놓는 것은 호출 계획이고, 계산은 파이썬이 함**
- `args`가 딕셔너리라 `add.invoke({'aaa': 21, 'bbb': 32})` 꼴로 그대로 들어감
- `tool_calls`가 비면 for문이 한 번도 돌지 않아서 아무것도 찍히지 않음

#### cf) 추론에 밀려 잘리는 도구 호출

- Qwen3는 답하기 전에 `<think>`로 생각을 먼저 적고, 그 글자도 `max_tokens`에 들어감
- `max_tokens`가 모자라면 생각 중간에서 끊겨 도구 호출을 적지 못하고 `tool_calls`가 빈 목록이 됨
- `response.response_metadata`의 `finish_reason`이 `'length'`면 토큰 수에 걸려 끊긴 것. 다 적고 끝나면 `'tool_calls'`
- 넣는 숫자가 커지면 생각도 길어져서, 코드를 그대로 두고 숫자만 바꿔도 결과가 달라짐